# PPO Fine-tuning — Kaggle

Дообучает политику через PPO начиная с BC-весов.

**Перед запуском:**
1. Загрузи `bc_pretrained.zip` на Kaggle Datasets
2. Добавь датасет через Add Input
3. Accelerator: T4 x2 (используется одна T4)
4. Internet: On

In [ ]:
# Клонируем репо и устанавливаем зависимости
# torch не указываем — на Kaggle уже установлен
!git clone https://github.com/Andrew82mm/RL_practice.git /kaggle/working/RL_practice
%cd /kaggle/working/RL_practice
!pip install -q sb3-contrib gymnasium scipy tqdm

In [ ]:
import IPython
IPython.Application.instance().kernel.do_shutdown(True)  # перезапуск после pip install

In [ ]:
import torch, os
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'не найден!'}")
%cd /kaggle/working/RL_practice

In [ ]:
# Находим bc_pretrained.zip в датасете
import os, shutil

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if "bc_pretrained" in f:
            src = os.path.join(root, f)
            print(f"Найдено: {src}")

# Копируем в рабочую папку
BC_SRC = src  # путь из вывода выше
BC_PATH = "/kaggle/working/bc_pretrained.zip"
shutil.copy2(BC_SRC, BC_PATH)
print(f"Скопировано: {BC_PATH}")

In [ ]:
# Запускаем PPO с BC-инициализацией
# --timesteps 15000000  — 15M шагов (~2-3 часа на T4)
# --n-envs 8           — меньше чем 16 чтобы не перегружать 2 CPU ядра Kaggle
!python training/run_training.py \
    --arch cnn \
    --bc-pretrained /kaggle/working/bc_pretrained.zip \
    --timesteps 15000000 \
    --n-envs 8 \
    --run-name cnn_bc_finetune

In [ ]:
# Проверяем сохранённые модели
for root, dirs, files in os.walk("/kaggle/working/models"):
    for f in files:
        path = os.path.join(root, f)
        print(f"{path}  ({os.path.getsize(path)/1024**2:.1f} MB)")